# Class-Based Views (CBVs)

## CBVs vs Function-Based Views (FBVs)

| FBV | CBV |
|-----|-----|
| Easy to read for simple cases | Structured; separates GET and POST |
| Logic for all methods in one function | Clear `get()` and `post()` methods |
| Code often duplicated across views | Reusable through inheritance and mixins |
| Good for simple views | Better for complex or repetitive views |

CBVs solve the problem of mixing GET and POST logic together, and make it easy to share behavior across views.


## Signup and Login with `View`

```python
from django.views import View
from django.shortcuts import render, redirect
from django.contrib.auth import authenticate, login
from .models import CustomUser

class SignupView(View):
    def get(self, request):
        return render(request, 'signup.html')

    def post(self, request):
        mobile = request.POST['mobile']
        password = request.POST['password']
        user = CustomUser.objects.create_user(mobile=mobile, password=password)
        login(request, user)
        return redirect('home')

class LoginView(View):
    def get(self, request):
        return render(request, 'login.html')

    def post(self, request):
        mobile = request.POST['mobile']
        password = request.POST['password']
        user = authenticate(request, mobile=mobile, password=password)
        if user:
            login(request, user)
            return redirect('home')
        return render(request, 'login.html', {'error': 'Invalid credentials'})
```


## TemplateView and RedirectView

### TemplateView — render a template, optionally with context
```python
from django.views.generic import TemplateView

class HomeView(TemplateView):
    template_name = 'home.html'

    def get_context_data(self, **kwargs):
        context = super().get_context_data(**kwargs)
        context['message'] = 'Welcome!'
        return context
```

### RedirectView — redirect from one URL to another
```python
from django.views.generic import RedirectView

# In urls.py:
path('old-path/', RedirectView.as_view(url='/new-path/'), name='old_path'),
```


## ListView and DetailView

```python
from django.views.generic import ListView, DetailView
from .models import Profile

class ProfileListView(ListView):
    model = Profile
    template_name = 'profile_list.html'
    context_object_name = 'profiles'

class ProfileDetailView(DetailView):
    model = Profile
    template_name = 'profile_detail.html'
```

Templates can access the list as `{{ profiles }}` and the single object as `{{ object }}`.


## Manual Create, Update, and Delete Views

```python
from django.contrib.auth.mixins import LoginRequiredMixin

class ProfileCreateView(LoginRequiredMixin, View):
    login_url = '/login/'

    def get(self, request):
        return render(request, 'profile_create.html')

    def post(self, request):
        bio = request.POST['bio']
        avatar = request.FILES.get('avatar')
        Profile.objects.create(user=request.user, bio=bio, avatar=avatar)
        return redirect('profile_list')

class ProfileUpdateView(LoginRequiredMixin, View):
    login_url = '/login/'

    def get(self, request, pk):
        profile = Profile.objects.get(pk=pk)
        return render(request, 'profile_update.html', {'profile': profile})

    def post(self, request, pk):
        profile = Profile.objects.get(pk=pk)
        profile.bio = request.POST['bio']
        if request.FILES.get('avatar'):
            profile.avatar = request.FILES['avatar']
        profile.save()
        return redirect('profile_detail', pk=pk)

class ProfileDeleteView(LoginRequiredMixin, View):
    login_url = '/login/'

    def get(self, request, pk):
        profile = Profile.objects.get(pk=pk)
        return render(request, 'profile_confirm_delete.html', {'profile': profile})

    def post(self, request, pk):
        Profile.objects.get(pk=pk).delete()
        return redirect('profile_list')
```


## LoginRequiredMixin

Add `LoginRequiredMixin` as the first base class to protect any CBV:

```python
from django.contrib.auth.mixins import LoginRequiredMixin

class ProtectedView(LoginRequiredMixin, View):
    login_url = '/login/'
    ...
```

### Built-in Generic CBVs (with Django Forms)

Once Django Forms are introduced, these provide the full create/update/delete workflow in very few lines:

```python
from django.views.generic.edit import CreateView, UpdateView, DeleteView

class ProfileCreateView(LoginRequiredMixin, CreateView):
    model = Profile
    fields = ['bio', 'avatar']
    template_name = 'profile_create.html'
    success_url = '/profiles/'
```


## URL Configuration

```python
from django.urls import path
from django.views.generic import RedirectView
from .views import (
    HomeView, SignupView, LoginView,
    ProfileListView, ProfileDetailView,
    ProfileCreateView, ProfileUpdateView, ProfileDeleteView,
)

urlpatterns = [
    path('', HomeView.as_view(), name='home'),
    path('signup/', SignupView.as_view(), name='signup'),
    path('login/', LoginView.as_view(), name='login'),
    path('profiles/', ProfileListView.as_view(), name='profile_list'),
    path('profiles/<int:pk>/', ProfileDetailView.as_view(), name='profile_detail'),
    path('profiles/create/', ProfileCreateView.as_view(), name='profile_create'),
    path('profiles/<int:pk>/update/', ProfileUpdateView.as_view(), name='profile_update'),
    path('profiles/<int:pk>/delete/', ProfileDeleteView.as_view(), name='profile_delete'),
    path('go-home/', RedirectView.as_view(url='/'), name='go_home'),
]
```


## Summary

- CBVs separate HTTP method handling into `get()` and `post()` methods.
- `TemplateView` renders a template; `RedirectView` redirects to another URL.
- `ListView` and `DetailView` handle list and single-object display.
- `LoginRequiredMixin` must be listed first in the base class list.
- Use `request.FILES` for file uploads (requires `enctype="multipart/form-data"` on the form).
- Built-in generic edit views (`CreateView`, `UpdateView`, `DeleteView`) integrate with Django Forms.
